# 33류 해설서 추출하여 텍스트 파일로 저장

In [1]:
import requests

URL = 'https://unipass.customs.go.kr/clip/index.do'
response = requests.get(URL)
print(response.status_code)

200


In [2]:
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from bs4 import BeautifulSoup
import time
import re

total_explanation = {}         # 모든 HS 품목 류 담을 딕셔너리
sub_explanation = {}           # 특정 류, 호 담을 딕셔리리

driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=webdriver.ChromeOptions())
driver.get('https://unipass.customs.go.kr/clip/index.do')
ele = driver.find_element(by=By.ID, value='TOPMENU_LNK_M_ULS0200000000')
ele.send_keys(Keys.ENTER)
time.sleep(3)

# 33류 사이트 접속
ele = driver.find_element(by=By.CSS_SELECTOR, value='#tblLstBody > tr:nth-child(4) > td:nth-child(5) > a')
ele.send_keys(Keys.ENTER)

In [3]:
# 류 해설서
ryu = driver.find_element(by=By.CSS_SELECTOR, value='#divLft_tab3 > pre').text
# ryu = re.sub('[※ㅇ\-•●◆▶★]', '', ryu)                      # 특수기호 제거
ryu = re.sub(r'\(([\u4E00-\u9FFF]{1,10})\)', '', ryu)      # 한자 제거
# ryu = ryu.replace('\n', ' ')                               # 개행문자 제거
sub_explanation['ryu'] = ryu

In [4]:
# 호 해설서
ho_ex_dict = {}
cnt = len(driver.find_elements(by=By.CSS_SELECTOR, value='#tblLstBody > tr'))
for i in range(1,cnt+1):
    value = '#tblLstBody > tr:nth-child(%d) > td:nth-child(2) > a' %i
    ho = driver.find_element(by=By.CSS_SELECTOR, value=value)
    ho.click()
    time.sleep(3)
    ho_ex = driver.find_element(by=By.CSS_SELECTOR, value='#divLft_tab4 > pre').text
    # ho_ex = re.sub('[※ㅇ\-•●◆▶★]', '', ho_ex)                      # 특수기호 제거
    ho_ex = re.sub(r'\(([\u4E00-\u9FFF]{1,10})\)', '', ho_ex)      # 한자 제거
    # ho_ex = ho_ex.replace('\n', ' ')                               # 개행문자 거거
    ho_ex_dict[i] = ho_ex
    driver.back()
    time.sleep(2)
sub_explanation['ho'] = ho_ex_dict

In [5]:
# 텍스트 파일로 저장하기
with open('./HS_explanation_33.txt', 'w', encoding='utf-8') as f:
    f.write(str(sub_explanation))

# 33류 해설서 추출하여 csv 파일로 저장

In [5]:
import pandas as pd

df_33 = pd.DataFrame(sub_explanation)

In [6]:
# 류 번호, 류 해설, 호 번호, 호 해설 분리하기
df_33['ryu_ex'] = df_33['ryu'].str[5:]   # 류 해설
df_33['ryu'] = df_33['ryu'].str[1:3]     # 류 번호
df_33['ho_ex'] = df_33['ho'].str[6:]     # 호 해설
df_33['ho'] = df_33['ho'].str[3:5]       # 호 번호

In [7]:
# HS_4 컬럼 추가
df_33['HS_4'] = df_33['ryu'] + df_33['ho']

# 컬럼 정렬
df_33 = df_33.reindex(columns=['HS_4', 'ryu', 'ryu_ex', 'ho', 'ho_ex'])

In [10]:
# csv 파일 저장하기
df_33.to_csv('./dataset/HS_33_explanation.csv', index=False)

In [8]:
df_33.head()

,HS_4,ryu,ryu_ex,ho,ho_ex
1,3301,33,"정유(essential oil)와 레지노이드(resinoid), 조제향료와 화장품ㆍ...",01,- 정유(essential oil)[콘크리트(concrete)와 앱설루트(absol...
2,3302,33,"정유(essential oil)와 레지노이드(resinoid), 조제향료와 화장품ㆍ...",02,- 방향성 물질의 혼합물과 방향성 물질의 하나 이상을 기본 재료로 한 혼합물(알코올...
3,3303,33,"정유(essential oil)와 레지노이드(resinoid), 조제향료와 화장품ㆍ...",03,- 향수와 화장수\n이 호에는 주로 인체에 향기를 주도록 고안된 액체 상태ㆍ크림 상...
4,3304,33,"정유(essential oil)와 레지노이드(resinoid), 조제향료와 화장품ㆍ...",04,"- 미용이나 메이크업용 제품류와 기초화장용 제품류[의약품은 제외하며, 선스크린(su..."
5,3305,33,"정유(essential oil)와 레지노이드(resinoid), 조제향료와 화장품ㆍ...",05,- 두발용 제품류\n3305.10 - 샴푸\n3305.20 - 퍼머넌트 웨이빙(pe...
